# Module 322 — LLM Safety Tuning

This notebook is a safe, annotated companion to the adversarial-tuning lessons. It demonstrates the loss and LoRA arithmetic locally without downloading a model or running harmful prompts. Full fine-tuning is intentionally opt-in.

In [ ]:
import math

# Negative log-likelihood for three correct next-token probabilities.
probabilities = [0.8, 0.5, 0.9]
nll = -sum(math.log(p) for p in probabilities)
print(f"sequence NLL: {nll:.4f}")
assert nll > 0


For `L = -Σ log P(x_i | previous tokens; theta)`, read `Σ` as **sum**, `x_i` as **x sub i**, and the vertical bar `|` as **given**. If the model assigns probability 0.8, 0.5, and 0.9 to the correct next tokens, the loss is `-(log .8 + log .5 + log .9)`. Raising any of those probabilities lowers the loss.

In [ ]:
# LoRA: W-prime = W + B @ A. Only A and B are trainable.
import numpy as np

d, k, rank = 4, 3, 1
W = np.zeros((d, k))
B = np.ones((d, rank))
A = np.ones((rank, k))
W_prime = W + B @ A
assert W_prime.shape == (d, k)
print("full parameters:", d * k)
print("adapter parameters:", rank * (d + k))


Training data should balance jailbreak refusals, priming-defense examples, and benign conversations. Evaluate both attack refusal and benign helpfulness; optimizing only refusal produces over-refusal. A real model-training cell should be added only after the course specifies the exact base model, tokenizer, dataset files, and hardware budget.

## Training configuration without launching a model

For the course run, four-bit loading reduces frozen-weight memory, LoRA rank 16 limits trainable updates, and gradient accumulation gives an effective batch of 8. This cell records the arithmetic only; it intentionally does not download Llama or start GPU training.

In [ ]:
per_device_batch_size = 4
gradient_accumulation_steps = 2
effective_batch_size = per_device_batch_size * gradient_accumulation_steps
warmup_ratio = 0.1
trainable = 11_272_192
total = 785_713_152
print("effective batch:", effective_batch_size)
print("trainable percentage:", round(100 * trainable / total, 2))
assert effective_batch_size == 8
assert 0 < warmup_ratio < 1
